# NFL Elo Rating Model

Prospective NFL Elo ratings with margin-of-victory adjustment, home-field advantage, offseason regression, and expanding-window parameter tuning.


In [ ]:
# pip install nflreadpy pandas pyarrow


  Using cached nflreadpy-0.1.5-py3-none-any.whl.metadata (7.5 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached polars-1.44.2-py3-none-any.whl.metadata (11 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached pydantic_settings-2.15.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached polars_runtime_32-1.44.2-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached nflreadpy-0.1.5-py3-none-any.whl (30 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -------------- ------------------------- 3.4/9.7 MB 18.3 MB/s eta 0:00:01
   -------------------------- ------------- 6.3/9.7 MB 15.4 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.7 MB 15.0 MB/s eta 0:00:01
   ----------

## 1. Setup and NFL Data


In [ ]:
import numpy as np 
import pandas as pd
import nflreadpy as nfl
from sklearn.metrics import accuracy_score, log_loss

#Load games from nfl library
games = nfl.load_schedules(seasons=True).to_pandas()
print(games.columns)
print(games.shape)
print(games.head())


Index(['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday',
       'gametime', 'away_team', 'away_score', 'home_team', 'home_score',
       'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis',
       'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest',
       'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds',
       'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game',
       'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id',
       'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee',
       'stadium_id', 'stadium'],
      dtype='str')
(7548, 46)
           game_id  season game_type  week     gameday weekday gametime  \
0  1999_01_MIN_ATL    1999       REG     1  1999-09-12  Sunday      NaN   
1   1999_01_KC_CHI    1999       REG     1  1999-09-12  Sunday      NaN   
2  1999_01_PIT_CLE    1999       REG     1  1999-09-12  Sunday      NaN   
3   1999_01_OAK_GB    1999       RE

In [5]:
games['location'].unique()

<ArrowStringArray>
['Home', 'Neutral']
Length: 2, dtype: str

In [6]:
games['season'].unique()

array([1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009,
       2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020,
       2021, 2022, 2023, 2024, 2025, 2026], dtype=int32)

## 2. Core Elo Rating Functions


In [7]:
import numpy as np
import pandas as pd
from IPython.display import display


def null_coalesce(x, y):
    return y if x is None else x

def compress_exp_grid(min_val, max_val, n=10, curve=2):
    grid = np.unique(np.round(
        min_val + (max_val - min_val) *
        (np.exp(curve * np.linspace(0, 1, n)) - 1) / (np.exp(curve) - 1)
    ))
    return grid

def calc_mov_scale(games):
    mean_log_margin = np.log(
        (games["home_score"] - games["away_score"]).abs() + 1
    ).mean()
    return mean_log_margin

def nfl_home_result(home_score, away_score, overtime):
    conditions = [
        (home_score > away_score) & (overtime == 0),
        (home_score < away_score) & (overtime == 0),
        (home_score > away_score) & (overtime == 1),
        (home_score < away_score) & (overtime == 1),
        home_score == away_score
    ]
    choices = [1.00, 0.00, 0.75, 0.25, 0.50]
    return np.select(conditions, choices, default=np.nan)

def elo_season(games, K, ratings=None, scale=400, start_rating=1000,
               use_mov=True, mov_scale=1, use_home_adv=True, home_adv=0):

    ratings = {} if ratings is None else ratings.copy()
    n = len(games)

    if use_mov and (pd.isna(mov_scale) or mov_scale <= 0):
        mov_scale = 1
    if not use_home_adv:
        home_adv = 0

    elo_home_pre = np.zeros(n)
    elo_away_pre = np.zeros(n)
    elo_home_post = np.zeros(n)
    elo_away_post = np.zeros(n)
    p_home = np.zeros(n)

    for i in range(n):
        g = games.iloc[i]

        hid = str(g["home_team"])
        aid = str(g["away_team"])

        Ra = ratings.get(hid, start_rating)
        Rb = ratings.get(aid, start_rating)

        elo_home_pre[i] = Ra
        elo_away_pre[i] = Rb

        game_home_adv = 0 if g["location"] == "Neutral" else home_adv

        p = 1 / (1 + 10 ** ((Rb - Ra - game_home_adv) / scale))
        p_home[i] = p

        y = nfl_home_result(home_score=g["home_score"], away_score=g["away_score"], overtime=g["overtime"])

        if pd.isna(y):
            raise ValueError(f"Invalid NFL result coding for game_id: {g['game_id']}")

        point_diff = abs(g["home_score"] - g["away_score"])
        if g["home_score"] > g["away_score"]:
            winner_elo_diff = (Ra + game_home_adv) - Rb
        elif g["home_score"] < g["away_score"]:
            winner_elo_diff = Rb - (Ra + game_home_adv)
        else:
            winner_elo_diff = 0

        if not use_mov:
          mov_mult = 1
        elif g["home_score"] == g["away_score"]:
          mov_mult = 1
        else:
          mov_mult = np.log(point_diff + 1) * (2.2 / (0.001 * winner_elo_diff + 2.2)) / mov_scale

        delta = K * (y - p) * mov_mult

        ratings[hid] = Ra + delta
        ratings[aid] = Rb - delta

        elo_home_post[i] = ratings[hid]
        elo_away_post[i] = ratings[aid]

    games_out = games.copy()
    games_out["elo_home_pre"] = elo_home_pre
    games_out["elo_away_pre"] = elo_away_pre
    games_out["elo_home_post"] = elo_home_post
    games_out["elo_away_post"] = elo_away_post
    games_out["p_home"] = p_home
    games_out["p_away"] = 1 - p_home

    return {"games": games_out, "ratings": ratings}

### Single-Season Elo Sanity Check


In [8]:
# Sort it chronologically, calculate its MOV scale, and run
test_games = games[(games["season"] == 2024) & games["home_score"].notna() & games["away_score"].notna()].copy()

test_games = test_games.sort_values(["gameday", "game_id"]).reset_index(drop=True)
mov_scale = calc_mov_scale(test_games)
test_elo = elo_season(games=test_games, K=20, ratings={}, scale=400, start_rating=1000, use_mov=True, mov_scale=mov_scale, use_home_adv=True, home_adv=50)

test_elo["games"][["gameday", "home_team", "away_team", "elo_home_pre", "elo_away_pre", "p_home", "elo_home_post", "elo_away_post"]].head(20)
display(test_elo["ratings"])

#Sanity Check - average of elo values should still be around starting mean of 1000
print("Average ELO Rank: ",
      sum(test_elo["ratings"].values())
      / len(test_elo["ratings"]))

{'KC': np.float64(1080.1455240974985),
 'BAL': np.float64(1084.99145629367),
 'PHI': np.float64(1139.394535729747),
 'GB': np.float64(1047.9716492221542),
 'BUF': np.float64(1091.284579941748),
 'ARI': np.float64(990.394293099358),
 'NO': np.float64(940.3767154879677),
 'CAR': np.float64(916.9178851408424),
 'CLE': np.float64(900.6120500589781),
 'DAL': np.float64(961.0226239265742),
 'SEA': np.float64(1017.0399562777417),
 'DEN': np.float64(1040.6432159452693),
 'IND': np.float64(967.2166883269182),
 'HOU': np.float64(1019.2687958528423),
 'MIA': np.float64(988.8597186352837),
 'JAX': np.float64(937.0372823321698),
 'DET': np.float64(1098.2585227463162),
 'LA': np.float64(1025.6694376404287),
 'LAC': np.float64(1040.0149004296873),
 'LV': np.float64(918.7376747902182),
 'NYG': np.float64(908.4340687048452),
 'MIN': np.float64(1058.9723201568206),
 'CIN': np.float64(1024.1691489524187),
 'NE': np.float64(932.7520216866079),
 'ATL': np.float64(979.5503642546234),
 'PIT': np.float64(1012

Average ELO Rank:  1000.0000000000003


## 3. Offseason Regression and Evaluation


In [9]:
def carry_over(ratings, carry=0.6):
    if len(ratings) == 0:
        return ratings

    mu = np.mean(list(ratings.values()))
    return {team: carry * rating + (1 - carry) * mu
            for team, rating in ratings.items()}

def logloss(p, y, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(y * np.log(p) + (1 - y) * np.log1p(-p))

### Carryover Sanity Check


In [10]:
#Sanity Checks
test_carry = carry_over(test_elo["ratings"], carry=0.75)

print(np.mean(list(test_carry.values())))
print(max(test_elo["ratings"].values()))
print(max(test_carry.values()))

1000.0
1139.394535729747
1104.5459017973103


## 4. Elo Parameter Evaluation


In [11]:
def score_elo_params(all_games, seasons_tune, K, home_adv,
                     use_mov=True, mov_scale=1, use_home_adv=True,
                     scale=400, start_rating=1000, carry=0.6):

  ratings = {}
  ll_vec = []

  if not use_home_adv:
      home_adv = 0

  for s in seasons_tune:
      g_s = (all_games[all_games["season"] == s]
              .sort_values(["gameday", "game_id"])
              .reset_index(drop=True))

      res = elo_season(games=g_s, ratings=ratings, K=K, scale=scale,
                       start_rating=start_rating, use_mov=use_mov,
                       mov_scale=mov_scale, use_home_adv=use_home_adv,
                       home_adv=home_adv)

      games_res = res["games"]
      non_ties = games_res["home_score"] != games_res["away_score"]

      p = games_res.loc[non_ties, "p_home"].to_numpy()
      y = (games_res.loc[non_ties, "home_score"] >
            games_res.loc[non_ties, "away_score"]).astype(float).to_numpy()

      ll_vec.extend(logloss(p, y))
      ratings = carry_over(res["ratings"], carry=carry)

  return np.nanmean(ll_vec)

### Initial Parameter Test


In [12]:
test_score = score_elo_params(
    all_games=games,
    seasons_tune=[2021, 2022, 2023, 2024],
    K=20,
    home_adv=50,
    use_mov=True,
    mov_scale=calc_mov_scale(games[games["season"].isin([2021, 2022, 2023, 2024])]),
    use_home_adv=True,
    scale=400,
    start_rating=1000,
    carry=0.75
)

print(test_score)

0.6509180246215944


$0.6503$ is already meaningfully better than a naive 50/50 baseline:
$−log(.5)=0.6931$


## 5. Hyperparameter Tuning


In [13]:
from itertools import product

def tune_elo_params(all_games, seasons_tune, K_grid, home_adv_grid,
                    use_mov=True, mov_scale=1, use_home_adv=True,
                    scale=400, start_rating=1000, carry=0.6,
                    use_carry_grid=True,
                    carryover_grid=(0.35, 0.50, 0.65, 0.80)):

    if not use_home_adv:
        home_adv_grid = [0]

    carry_grid = carryover_grid if use_carry_grid else [carry]
    results = []

    for K, home_adv, carry_val in product(K_grid, home_adv_grid, carry_grid):
        ll = score_elo_params(
            all_games=all_games, seasons_tune=seasons_tune,
            K=K, home_adv=home_adv,
            use_mov=use_mov, mov_scale=mov_scale,
            use_home_adv=use_home_adv, scale=scale,
            start_rating=start_rating, carry=carry_val
        )

        results.append({"K": K, "home_adv": home_adv, "carry": carry_val,
                        "logloss": ll})

    results = pd.DataFrame(results).sort_values("logloss").reset_index(drop=True)

    best = results.iloc[0]

    return {"K_best": best["K"], "home_adv_best": best["home_adv"],
            "carry_best": best["carry"], "logloss": best["logloss"],
            "results": results}

### Tuning Sanity Check


In [14]:
K_grid = [10, 15, 20, 25]
home_adv_grid = [25, 40, 55, 70]

mov_scale = calc_mov_scale(games[games["season"].isin([2021, 2022, 2023, 2024])])

test_tune = tune_elo_params(all_games=games, seasons_tune=[2021, 2022, 2023, 2024], K_grid=K_grid, 
                            home_adv_grid=home_adv_grid, mov_scale=mov_scale, carryover_grid=[0.50, 0.65, 0.75, 0.85])

print(test_tune["K_best"])
print(test_tune["home_adv_best"])
print(test_tune["carry_best"])
print(test_tune["logloss"])

test_tune["results"].head(10)

25.0
25.0
0.85
0.646844717404286


,K,home_adv,carry,logloss
0,25,25,0.85,0.646845
1,25,40,0.85,0.646937
2,25,25,0.75,0.646945
3,25,40,0.75,0.647051
4,25,25,0.65,0.647522
5,25,40,0.65,0.647638
6,25,55,0.85,0.648839
7,25,55,0.75,0.648971
8,25,25,0.50,0.649085
9,25,40,0.50,0.649212


In [16]:
compress_exp_grid(1, 50, n=10, curve=-1)
compress_exp_grid(1, 60, n=10, curve=1)


array([ 1.,  5., 10., 15., 20., 27., 34., 41., 50., 60.])

## 6. Prospective Expanding-Window Elo


In [17]:
def run_expanding_k_elo(all_games, first_train_year=2003, first_eval_year=2010,
                        K_grid=None, home_adv_grid=None, scale=400,
                        start_rating=1000, carry=0.6, use_carry_grid=True,
                        carryover_grid=(0.35, 0.50, 0.65, 0.80),
                        param_tune_window=None, use_mov=True, use_home_adv=True,
                        projection_season=None):

    if K_grid is None:
        K_grid = compress_exp_grid(1, 60, n=10, curve=-1)
    if home_adv_grid is None:
        home_adv_grid = compress_exp_grid(1, 60, n=10, curve=1)
    if not use_home_adv:
        home_adv_grid = [0]

    K_grid = np.atleast_1d(K_grid)
    home_adv_grid = np.atleast_1d(home_adv_grid)
    fixed_param_mode = len(K_grid) == 1 and len(home_adv_grid) == 1 and not use_carry_grid

    seasons = sorted(all_games["season"].unique())
    eval_years = [s for s in seasons if s >= first_eval_year]

    # Include the current schedule season even if it has no completed games yet.
    if projection_season is not None and projection_season not in eval_years:
        eval_years = sorted(set(eval_years + [projection_season]))

    all_out = []
    param_out = []

    for yr in eval_years:
        train_years = [s for s in seasons if first_train_year <= s < yr]
        if len(train_years) == 0:
            raise ValueError(f"No training seasons available before eval year: {yr}")

        param_tune_years = (
            train_years if param_tune_window is None
            else train_years[-param_tune_window:]
        )

        train_games = all_games[all_games["season"].isin(train_years)]
        mov_scale = calc_mov_scale(train_games) if use_mov else np.nan

        if fixed_param_mode:
            K_best = K_grid[0]
            home_adv_best = home_adv_grid[0] if use_home_adv else 0
            carry_best = carry
            best_logloss = np.nan
        else:
            param_fit = tune_elo_params(
                all_games=all_games, seasons_tune=param_tune_years,
                K_grid=K_grid, home_adv_grid=home_adv_grid,
                use_mov=use_mov, mov_scale=mov_scale,
                use_home_adv=use_home_adv, scale=scale,
                start_rating=start_rating, carry=carry,
                use_carry_grid=use_carry_grid,
                carryover_grid=carryover_grid
            )

            K_best = param_fit["K_best"]
            home_adv_best = param_fit["home_adv_best"] if use_home_adv else 0
            carry_best = param_fit["carry_best"] if use_carry_grid else carry
            best_logloss = param_fit["logloss"]

        ratings = {}

        for s in train_years:
            season_games = (all_games[all_games["season"] == s]
                            .sort_values(["gameday", "gametime", "game_id"])
                            .reset_index(drop=True))

            res = elo_season(
                games=season_games, ratings=ratings, K=K_best,
                scale=scale, start_rating=start_rating,
                use_mov=use_mov, mov_scale=mov_scale,
                use_home_adv=use_home_adv, home_adv=home_adv_best
            )

            ratings = carry_over(res["ratings"], carry=carry_best)

        eval_games = (all_games[all_games["season"] == yr]
                      .sort_values(["gameday", "gametime", "game_id"])
                      .reset_index(drop=True))

        res_eval = elo_season(
            games=eval_games, ratings=ratings, K=K_best,
            scale=scale, start_rating=start_rating,
            use_mov=use_mov, mov_scale=mov_scale,
            use_home_adv=use_home_adv, home_adv=home_adv_best
        )

        games_out = res_eval["games"].copy()
        games_out["K_used"] = K_best
        games_out["home_adv_used"] = np.where(games_out["location"] == "Neutral", 0, home_adv_best)
        games_out["carry_used"] = carry_best
        games_out["mov_scale_used"] = mov_scale
        games_out["use_mov"] = use_mov
        games_out["use_home_adv"] = use_home_adv
        games_out["train_start"] = min(train_years)
        games_out["train_end"] = max(train_years)
        games_out["param_tune_start"] = min(param_tune_years)
        games_out["param_tune_end"] = max(param_tune_years)
        games_out["param_tune_n"] = len(param_tune_years)
        games_out["param_tune_window"] = param_tune_window

        all_out.append(games_out)

        param_out.append({
            "eval_year": yr,
            "train_start": min(train_years),
            "train_end": max(train_years),
            "param_tune_start": min(param_tune_years),
            "param_tune_end": max(param_tune_years),
            "param_tune_n": len(param_tune_years),
            "param_tune_window": param_tune_window,
            "K_best": K_best,
            "home_adv_best": home_adv_best,
            "carry_best": carry_best,
            "mov_scale": mov_scale,
            "use_mov": use_mov,
            "use_home_adv": use_home_adv,
            "use_carry_grid": use_carry_grid,
            "logloss": best_logloss
        })

    return {
        "games": pd.concat(all_out, ignore_index=True),
        "K_by_year": pd.DataFrame(param_out),
        "ratings": res_eval["ratings"],
        "current_season": eval_years[-1]
    }

### Prospective Validation Test


In [18]:
# Exclude Preseason and Unplayed Games
elo_df = games[games["game_type"].isin(["REG", "WC", "DIV", "CON", "SB"]) &
               games["home_score"].notna() &
               games["away_score"].notna()].copy()

test_expanding = run_expanding_k_elo(
    all_games=elo_df,
    first_train_year=2003,
    first_eval_year=2008,
    K_grid=[15, 20, 25],
    home_adv_grid=[25, 40, 55],
    scale=400,
    start_rating=1000,
    use_carry_grid=True,
    carryover_grid=[0.3, 0.5, 0.75, 0.90],
    param_tune_window=5,
    use_mov=True,
    use_home_adv=True
)

#Inspect
display(test_expanding["K_by_year"])
display(test_expanding["games"][["season", "gameday", "home_team", "away_team",
                         "elo_home_pre", "elo_away_pre", "p_home", "K_used",
                         "location", "home_adv_used", "carry_used", "mov_scale_used"]])

#Sanity Check
test_expanding["K_by_year"][["eval_year", "train_start", "train_end", "param_tune_start", "param_tune_end"]]

,eval_year,train_start,train_end,param_tune_start,param_tune_end,param_tune_n,param_tune_window,K_best,home_adv_best,carry_best,mov_scale,use_mov,use_home_adv,use_carry_grid,logloss
0,2008,2003,2007,2003,2007,5,5,25.0,55.0,0.75,2.275855,True,True,True,0.635605
1,2009,2003,2008,2004,2008,5,5,25.0,55.0,0.75,2.279179,True,True,True,0.636476
2,2010,2003,2009,2005,2009,5,5,25.0,55.0,0.90,2.290766,True,True,True,0.630419
3,2011,2003,2010,2006,2010,5,5,25.0,40.0,0.75,2.285598,True,True,True,0.636809
4,2012,2003,2011,2007,2011,5,5,25.0,55.0,0.75,2.286571,True,True,True,0.630816
5,2013,2003,2012,2008,2012,5,5,25.0,55.0,0.75,2.284832,True,True,True,0.631811
6,2014,2003,2013,2009,2013,5,5,25.0,55.0,0.75,2.278318,True,True,True,0.635176
7,2015,2003,2014,2010,2014,5,5,25.0,55.0,0.75,2.283531,True,True,True,0.629776
8,2016,2003,2015,2011,2015,5,5,25.0,55.0,0.75,2.278831,True,True,True,0.629248
9,2017,2003,2016,2012,2016,5,5,25.0,55.0,0.75,2.270475,True,True,True,0.629925


,season,gameday,home_team,away_team,elo_home_pre,elo_away_pre,p_home,K_used,location,home_adv_used,carry_used,mov_scale_used
0,2008,2008-09-04,NYG,WAS,1048.950030,1006.031632,0.637300,25.0,Home,55.0,0.75,2.275855
1,2008,2008-09-07,BAL,CIN,988.199608,1007.759599,0.550826,25.0,Home,55.0,0.75,2.275855
2,2008,2008-09-07,ATL,DET,913.696713,933.909699,0.549896,25.0,Home,55.0,0.75,2.275855
3,2008,2008-09-07,PIT,HOU,1061.559852,990.172086,0.674267,25.0,Home,55.0,0.75,2.275855
4,2008,2008-09-07,TEN,JAX,1009.503972,1068.254384,0.494603,25.0,Home,55.0,0.75,2.275855
...,...,...,...,...,...,...,...,...,...,...,...,...
4939,2026,2026-09-27,SF,ARI,1070.928347,926.045916,0.726696,25.0,Home,25.0,0.75,2.249689
4940,2026,2026-09-27,TB,MIN,975.368298,1065.382695,0.407514,25.0,Home,25.0,0.75,2.249689
4941,2026,2026-09-27,DAL,BAL,972.101698,1049.093943,0.390978,25.0,Neutral,0.0,0.75,2.249689
4942,2026,2026-09-27,NO,LV,958.548203,917.639532,0.593729,25.0,Home,25.0,0.75,2.249689


,eval_year,train_start,train_end,param_tune_start,param_tune_end
0,2008,2003,2007,2003,2007
1,2009,2003,2008,2004,2008
2,2010,2003,2009,2005,2009
3,2011,2003,2010,2006,2010
4,2012,2003,2011,2007,2011
5,2013,2003,2012,2008,2012
6,2014,2003,2013,2009,2013
7,2015,2003,2014,2010,2014
8,2016,2003,2015,2011,2015
9,2017,2003,2016,2012,2016


In [19]:
display(test_expanding["K_by_year"])
display(test_expanding["games"][["season", "gameday", "home_team", "away_team", "elo_home_pre", "elo_away_pre", "p_home", "K_used", "location", "home_adv_used", "carry_used", "mov_scale_used"]])

,eval_year,train_start,train_end,param_tune_start,param_tune_end,param_tune_n,param_tune_window,K_best,home_adv_best,carry_best,mov_scale,use_mov,use_home_adv,use_carry_grid,logloss
0,2008,2003,2007,2003,2007,5,5,25.0,55.0,0.75,2.275855,True,True,True,0.635605
1,2009,2003,2008,2004,2008,5,5,25.0,55.0,0.75,2.279179,True,True,True,0.636476
2,2010,2003,2009,2005,2009,5,5,25.0,55.0,0.90,2.290766,True,True,True,0.630419
3,2011,2003,2010,2006,2010,5,5,25.0,40.0,0.75,2.285598,True,True,True,0.636809
4,2012,2003,2011,2007,2011,5,5,25.0,55.0,0.75,2.286571,True,True,True,0.630816
5,2013,2003,2012,2008,2012,5,5,25.0,55.0,0.75,2.284832,True,True,True,0.631811
6,2014,2003,2013,2009,2013,5,5,25.0,55.0,0.75,2.278318,True,True,True,0.635176
7,2015,2003,2014,2010,2014,5,5,25.0,55.0,0.75,2.283531,True,True,True,0.629776
8,2016,2003,2015,2011,2015,5,5,25.0,55.0,0.75,2.278831,True,True,True,0.629248
9,2017,2003,2016,2012,2016,5,5,25.0,55.0,0.75,2.270475,True,True,True,0.629925


,season,gameday,home_team,away_team,elo_home_pre,elo_away_pre,p_home,K_used,location,home_adv_used,carry_used,mov_scale_used
0,2008,2008-09-04,NYG,WAS,1048.950030,1006.031632,0.637300,25.0,Home,55.0,0.75,2.275855
1,2008,2008-09-07,BAL,CIN,988.199608,1007.759599,0.550826,25.0,Home,55.0,0.75,2.275855
2,2008,2008-09-07,ATL,DET,913.696713,933.909699,0.549896,25.0,Home,55.0,0.75,2.275855
3,2008,2008-09-07,PIT,HOU,1061.559852,990.172086,0.674267,25.0,Home,55.0,0.75,2.275855
4,2008,2008-09-07,TEN,JAX,1009.503972,1068.254384,0.494603,25.0,Home,55.0,0.75,2.275855
...,...,...,...,...,...,...,...,...,...,...,...,...
4939,2026,2026-09-27,SF,ARI,1070.928347,926.045916,0.726696,25.0,Home,25.0,0.75,2.249689
4940,2026,2026-09-27,TB,MIN,975.368298,1065.382695,0.407514,25.0,Home,25.0,0.75,2.249689
4941,2026,2026-09-27,DAL,BAL,972.101698,1049.093943,0.390978,25.0,Neutral,0.0,0.75,2.249689
4942,2026,2026-09-27,NO,LV,958.548203,917.639532,0.593729,25.0,Home,25.0,0.75,2.249689


### Out-of-Sample Performance


In [20]:
#Calculate out-of-sample performance of 2023-2026
eval_games = test_expanding["games"].copy()
non_ties = eval_games["home_score"] != eval_games["away_score"]

eval_games["actual_home_win"] = (
    eval_games["home_score"] > eval_games["away_score"]).astype(int)

eval_games["pred_home_win"] = (eval_games["p_home"] >= 0.5).astype(int)

eval_games["logloss"] = np.nan
eval_games.loc[non_ties, "logloss"] = logloss(
    eval_games.loc[non_ties, "p_home"],
    eval_games.loc[non_ties, "actual_home_win"]
)

performance_by_year = (eval_games.loc[non_ties].groupby("season").agg(
    games=("game_id", "size"),
    accuracy=("pred_home_win",
              lambda x: (x == eval_games.loc[x.index, "actual_home_win"]).mean()),
    logloss=("logloss", "mean")).reset_index())

print(performance_by_year)

    season  games  accuracy   logloss
0     2008    266  0.627820  0.632922
1     2009    267  0.700375  0.599334
2     2010    267  0.621723  0.659942
3     2011    267  0.655431  0.622486
4     2012    266  0.612782  0.632738
5     2013    266  0.627820  0.627409
6     2014    266  0.684211  0.603904
7     2015    267  0.674157  0.644909
8     2016    265  0.618868  0.631591
9     2017    267  0.670412  0.625702
10    2018    265  0.626415  0.633877
11    2019    266  0.646617  0.645102
12    2020    268  0.641791  0.628080
13    2021    284  0.619718  0.646518
14    2022    282  0.627660  0.649191
15    2023    285  0.607018  0.655652
16    2024    285  0.670175  0.618009
17    2025    284  0.640845  0.643238
18    2026     47  0.659574  0.644101


In [21]:
overall_accuracy = (eval_games.loc[non_ties, "pred_home_win"] ==
                    eval_games.loc[non_ties, "actual_home_win"]).mean()

overall_logloss = eval_games.loc[non_ties, "logloss"].mean()

print("Accuracy:", overall_accuracy)
print("Log Loss:", overall_logloss)

Accuracy: 0.6430020283975659
Log Loss: 0.6336264707196096


In [22]:
accuracy_by_year = (
    eval_games.loc[non_ties]
    .groupby("season")
    .apply(lambda x: (x["pred_home_win"] == x["actual_home_win"]).mean())
    .reset_index(name="accuracy")
)

accuracy_by_year

,season,accuracy
0,2008,0.627820
1,2009,0.700375
2,2010,0.621723
3,2011,0.655431
4,2012,0.612782
5,2013,0.627820
6,2014,0.684211
7,2015,0.674157
8,2016,0.618868
9,2017,0.670412


## 7. Final Prospective Elo Model


In [23]:
nfl_elo = run_expanding_k_elo(all_games=elo_df, first_train_year=2003,
                    first_eval_year=2008,
                    K_grid = compress_exp_grid(15, 60, n=10, curve=-1),
                    home_adv_grid = compress_exp_grid(15, 60, n=10, curve=1),
                    scale=400, start_rating=1000, use_carry_grid=True,
                    carryover_grid = [0.5, 0.65, 0.75, 0.85],
                    param_tune_window=5, use_mov=True, use_home_adv=True)


### Performance Check


In [28]:
eval_games = nfl_elo["games"].query("home_score != away_score").copy()

eval_games["actual"] = (eval_games["home_score"] > eval_games["away_score"]).astype(int)
eval_games["pred"] = (eval_games["p_home"] >= 0.5).astype(int)
eval_games["correct"] = eval_games["pred"] == eval_games["actual"]
eval_games["ll"] = -(eval_games["actual"] * np.log(eval_games["p_home"]) + (1 - eval_games["actual"]) * np.log1p(-eval_games["p_home"]))

print("Accuracy:", accuracy_score(eval_games["actual"], eval_games["pred"]))
print("Log Loss:", log_loss(eval_games["actual"], eval_games["p_home"]))

Accuracy: 0.6474645030425964
Log Loss: 0.6294838946163249


In [29]:
performance_by_year = eval_games.groupby("season").agg(
    games=("game_id", "size"), accuracy=("correct", "mean"),
    logloss=("ll", "mean")).reset_index()

performance_by_year

,season,games,accuracy,logloss
0,2008,266,0.654135,0.623722
1,2009,267,0.689139,0.602720
2,2010,267,0.614232,0.658355
3,2011,267,0.655431,0.613793
4,2012,266,0.642857,0.622481
5,2013,266,0.639098,0.613226
6,2014,266,0.665414,0.603797
7,2015,267,0.651685,0.647949
8,2016,265,0.641509,0.622938
9,2017,267,0.655431,0.621433


### Tuned Parameter Selection


In [30]:
params = nfl_elo["K_by_year"].copy()
params[["eval_year", "K_best", "home_adv_best", "carry_best", "mov_scale", "logloss"]]

,eval_year,K_best,home_adv_best,carry_best,mov_scale,logloss
0,2008,57.0,60.0,0.50,2.275855,0.628582
1,2009,57.0,53.0,0.50,2.279179,0.628483
2,2010,53.0,53.0,0.65,2.290766,0.623444
3,2011,45.0,46.0,0.65,2.285598,0.632569
4,2012,57.0,53.0,0.50,2.286571,0.623889
5,2013,53.0,53.0,0.50,2.284832,0.625189
6,2014,57.0,60.0,0.50,2.278318,0.626311
7,2015,57.0,60.0,0.50,2.283531,0.621984
8,2016,57.0,53.0,0.50,2.278831,0.622395
9,2017,57.0,60.0,0.50,2.270475,0.622580


In [31]:
print("K values:")
print(params["K_best"].value_counts().sort_index())

print("\nHome advantage values:")
print(params["home_adv_best"].value_counts().sort_index())

print("\nCarry values:")
print(params["carry_best"].value_counts().sort_index())
params[["K_best", "home_adv_best", "carry_best"]].describe()

K values:
K_best
41.0    1
45.0    3
50.0    6
53.0    2
57.0    7
Name: count, dtype: int64

Home advantage values:
home_adv_best
25.0    2
30.0    3
40.0    1
46.0    2
53.0    7
60.0    4
Name: count, dtype: int64

Carry values:
carry_best
0.50    11
0.65     7
0.75     1
Name: count, dtype: int64


,K_best,home_adv_best,carry_best
count,19.000000,19.000000,19.000000
mean,51.631579,46.473684,0.568421
std,5.090411,12.491635,0.085327
min,41.000000,25.000000,0.500000
25%,50.000000,35.000000,0.500000
50%,50.000000,53.000000,0.500000
75%,57.000000,53.000000,0.650000
max,57.000000,60.000000,0.750000


### Rerun Elo Tuning (Tightened Grids)


In [32]:
projection_season = int(games.loc[games["game_type"].isin(["REG", "WC", "DIV", "CON", "SB"]),"season"].max())

nfl_elo = run_expanding_k_elo(
    all_games=elo_df,
    first_train_year=2003,
    first_eval_year=2008,
    K_grid=compress_exp_grid(40, 65, n=8, curve=1),
    home_adv_grid=compress_exp_grid(25, 65, n=8, curve=1),
    scale=400,
    start_rating=1000,
    use_carry_grid=True,
    carryover_grid=[0.5, 0.575, 0.65],
    param_tune_window=5,
    use_mov=True,
    use_home_adv=True,
    projection_season=projection_season
)


In [33]:
eval_games = nfl_elo["games"].query("home_score != away_score").copy()
eval_games["actual"] = (eval_games["home_score"] > eval_games["away_score"]).astype(int)
eval_games["pred"] = (eval_games["p_home"] >= 0.5).astype(int)
eval_games["correct"] = eval_games["pred"] == eval_games["actual"]
eval_games["ll"] = -(eval_games["actual"] * np.log(eval_games["p_home"]) + (1 - eval_games["actual"]) * np.log1p(-eval_games["p_home"]))

print("Accuracy:", accuracy_score(eval_games["actual"], eval_games["pred"]))
print("Log Loss:", log_loss(eval_games["actual"], eval_games["p_home"]))

performance_by_year = eval_games.groupby("season").agg(
    games=("game_id", "size"), accuracy=("correct", "mean"),
    logloss=("ll", "mean")).reset_index()

performance_by_year

Accuracy: 0.6462474645030426
Log Loss: 0.629021369739824


,season,games,accuracy,logloss
0,2008,266,0.650376,0.623805
1,2009,267,0.685393,0.602794
2,2010,267,0.610487,0.656295
3,2011,267,0.662921,0.613154
4,2012,266,0.627820,0.623515
5,2013,266,0.650376,0.613639
6,2014,266,0.661654,0.603833
7,2015,267,0.655431,0.646147
8,2016,265,0.637736,0.623197
9,2017,267,0.670412,0.620406


In [39]:
params = nfl_elo["K_by_year"].copy()
params[["eval_year", "K_best", "home_adv_best", "carry_best", "mov_scale", "logloss"]].rename(columns = {"logloss":"tuning_logloss"})

,eval_year,K_best,home_adv_best,carry_best,mov_scale,tuning_logloss
0,2008,54.0,58.0,0.500,2.275855,0.628550
1,2009,57.0,52.0,0.500,2.279179,0.628483
2,2010,54.0,52.0,0.575,2.290766,0.623334
3,2011,48.0,47.0,0.575,2.285598,0.632519
4,2012,54.0,58.0,0.575,2.286571,0.623712
5,2013,51.0,58.0,0.575,2.284832,0.625162
6,2014,57.0,58.0,0.500,2.278318,0.626311
7,2015,54.0,58.0,0.575,2.283531,0.621890
8,2016,54.0,58.0,0.575,2.278831,0.622195
9,2017,54.0,58.0,0.575,2.270475,0.622417


## 8. Elo Win Probability to Spread Map

Validate the Elo-to-spread mapping prospectively with an expanding window.

For each evaluation season, the spread coefficient is fit only on completed games from earlier seasons, then applied to that season's already-prospective Elo win probabilities. This produces a leakage-safe historical spread validation.

After validation, a final production coefficient is fit on all completed historical games and used only for future projections.


In [40]:
from scipy.optimize import minimize_scalar

spread_games = nfl_elo["games"].dropna(subset=["home_score", "away_score", "p_home"]).copy()

spread_games = spread_games.sort_values(["season", "gameday", "gametime", "game_id"]).reset_index(drop=True)

spread_games["home_margin"] = (spread_games["home_score"] - spread_games["away_score"])

p = spread_games["p_home"].clip(1e-6, 1 - 1e-6)
spread_games["logit_p_home"] = np.log(p / (1 - p))

spread_seasons = sorted(spread_games["season"].unique())
spread_validation = []

for eval_season in spread_seasons[1:]:
    train = spread_games[spread_games["season"] < eval_season].copy()
    test = spread_games[spread_games["season"] == eval_season].copy()

    spread_opt = minimize_scalar(
        lambda slope: np.mean(
            np.abs(
                train["home_margin"]
                - slope * train["logit_p_home"]
            )
        ),
        bounds=(0, 20),
        method="bounded"
    )

    slope = spread_opt.x

    test["spread_slope_used"] = slope
    test["elo_home_margin"] = slope * test["logit_p_home"]
    test["model_home_spread"] = -(np.round(test["elo_home_margin"] * 2) / 2)
    test["spread_abs_error"] = (test["home_margin"] - test["elo_home_margin"]).abs()

    spread_validation.append(test)

spread_validation = pd.concat(spread_validation, ignore_index=True)

print("Prospective spread MAE:", spread_validation["spread_abs_error"].mean())
print("Fraction actual margin above predicted margin:", (spread_validation["home_margin"] > spread_validation["elo_home_margin"]).mean())


Prospective spread MAE: 10.444862373805357
Fraction actual margin above predicted margin: 0.5039555270472525


In [41]:
spread_performance_by_year = (spread_validation.groupby("season").agg(games=("game_id", "size"), spread_slope_used=("spread_slope_used", "first"),
                                                                      spread_mae=("spread_abs_error", "mean")).reset_index())

display(spread_performance_by_year)

,season,games,spread_slope_used,spread_mae
0,2009,267,6.330171,11.611845
1,2010,267,6.990437,11.071980
2,2011,267,6.735965,10.749526
3,2012,267,6.786008,11.064056
4,2013,267,6.669561,9.955455
5,2014,267,6.786006,11.327104
6,2015,267,6.751307,10.188257
7,2016,267,6.639251,9.379620
8,2017,267,6.560104,10.564133
9,2018,267,6.563575,10.288568


### Production Spread Coefficient

Fit one final coefficient on all completed historical games. This coefficient is not used to score the historical validation above; it is used only for future projections.


In [42]:
production_spread_opt = minimize_scalar(
    lambda slope: np.mean(
        np.abs(
            spread_games["home_margin"]
            - slope * spread_games["logit_p_home"]
        )
    ),
    bounds=(0, 20),
    method="bounded"
)

spread_slope = production_spread_opt.x

print(f"Production spread slope: {spread_slope:.4f}")


Production spread slope: 6.5643


## 9. Current Elo Ratings

Use the current ratings returned directly by the final expanding Elo fit. This keeps the rankings on the same 2018-forward training/replay path as the fitted model instead of separately replaying all available historical seasons with the current season's parameters.


In [43]:
current_season = nfl_elo["current_season"]
params = nfl_elo["K_by_year"].set_index("eval_year").loc[current_season]

# Use the final expanding fit's current Elo state directly.
ratings = nfl_elo["ratings"].copy()

current_teams = pd.unique(
    games.loc[
        games["season"] == current_season,
        ["home_team", "away_team"]
    ].values.ravel()
)

# Match the production script: a newly appearing team starts at 1000.
ratings = {team: ratings.get(team, 1000) for team in current_teams}

current_ratings = (
    pd.Series(ratings, name="elo")
    .rename_axis("team")
    .reset_index()
    .sort_values("elo", ascending=False)
    .reset_index(drop=True)
)
current_ratings.insert(0, "rank", range(1, len(current_ratings) + 1))

current_ratings


,rank,team,elo
0,1,SEA,1143.646958
1,2,BUF,1123.239631
2,3,SF,1104.125639
3,4,MIN,1097.746572
4,5,JAX,1090.737502
5,6,DEN,1074.899213
6,7,NE,1055.545798
7,8,PHI,1052.158872
8,9,LA,1048.959580
9,10,KC,1043.150809


## 10. Daily Rankings and Projections

Use current Elo ratings to identify games scheduled for a selected date and calculate each team's pregame Elo and win probability.


In [44]:
as_of_date = pd.Timestamp.now(tz="America/Chicago").date()
rank_map = current_ratings.set_index("team")["rank"]

future_projections = games[
    (games["season"] == current_season) &
    games["game_type"].isin(["REG", "WC", "DIV", "CON", "SB"]) &
    games["home_score"].isna() &
    games["away_score"].isna()
].copy()

future_projections["home_elo"] = future_projections["home_team"].map(ratings)
future_projections["away_elo"] = future_projections["away_team"].map(ratings)
future_projections["home_rank"] = future_projections["home_team"].map(rank_map)
future_projections["away_rank"] = future_projections["away_team"].map(rank_map)

future_projections["game_home_adv"] = np.where(future_projections["location"] == "Neutral", 0, params["home_adv_best"])
future_projections["p_home"] = 1 / (1 + 10 ** ((future_projections["away_elo"] - future_projections["home_elo"] - future_projections["game_home_adv"]) / 400))
future_projections["p_away"] = 1 - future_projections["p_home"]

projection_p = future_projections["p_home"].clip(1e-6, 1 - 1e-6)
future_projections["elo_home_margin"] = spread_slope * np.log(projection_p / (1 - projection_p))
future_projections["model_home_spread"] = -(np.round(future_projections["elo_home_margin"] * 2) / 2)
future_projections["as_of_date"] = as_of_date

future_projections = future_projections[["as_of_date", "week", "gameday", "gametime", "away_team", "home_team", "away_rank", "home_rank", "away_elo", "home_elo", "p_away", "p_home", "elo_home_margin", "model_home_spread"]
                                        ].sort_values(["gameday", "gametime"]).reset_index(drop=True)

daily_projections = future_projections[pd.to_datetime(future_projections["gameday"]).dt.date == as_of_date].reset_index(drop=True)

current_week = future_projections.iloc[0]["week"]
weekly_projections = future_projections[future_projections["week"] == current_week].reset_index(drop=True)

daily_rankings = current_ratings.assign(as_of_date=as_of_date)

display(future_projections)    # every remaining game using current Elo
display(daily_projections)     # today's games
display(weekly_projections)    # current/upcoming NFL week
display(daily_rankings)        # current league-wide rankings


,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home,elo_home_margin,model_home_spread
0,2026-09-28,3,2026-09-28,20:15,PHI,CHI,8,12,1052.158872,1030.539098,0.486504,0.513496,0.354454,-0.5
1,2026-09-28,4,2026-10-01,20:15,PIT,CLE,15,21,1014.731709,960.103159,0.533952,0.466048,-0.892861,1.0
2,2026-09-28,4,2026-10-04,09:30,IND,WAS,23,26,955.903550,949.149518,0.509719,0.490281,-0.255217,0.5
3,2026-09-28,4,2026-10-04,13:00,TEN,BAL,32,11,846.363273,1033.787538,0.221430,0.778570,8.253679,-8.5
4,2026-09-28,4,2026-10-04,13:00,NE,BUF,7,2,1055.545798,1123.239631,0.361669,0.638331,3.729381,-3.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,2026-09-28,18,2027-01-10,13:00,CHI,MIN,12,4,1030.539098,1097.746572,0.362316,0.637684,3.711002,-3.5
221,2026-09-28,18,2027-01-10,13:00,MIA,NE,31,7,891.656659,1055.545798,0.245664,0.754336,7.364348,-7.5
222,2026-09-28,18,2027-01-10,13:00,TB,NO,29,20,924.254191,961.643136,0.402832,0.597168,2.584238,-2.5
223,2026-09-28,18,2027-01-10,13:00,PHI,NYG,8,19,1052.158872,962.593912,0.583492,0.416508,-2.213016,2.0


,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home,elo_home_margin,model_home_spread
0,2026-09-28,3,2026-09-28,20:15,PHI,CHI,8,12,1052.158872,1030.539098,0.486504,0.513496,0.354454,-0.5


,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home,elo_home_margin,model_home_spread
0,2026-09-28,3,2026-09-28,20:15,PHI,CHI,8,12,1052.158872,1030.539098,0.486504,0.513496,0.354454,-0.5


,rank,team,elo,as_of_date
0,1,SEA,1143.646958,2026-09-28
1,2,BUF,1123.239631,2026-09-28
2,3,SF,1104.125639,2026-09-28
3,4,MIN,1097.746572,2026-09-28
4,5,JAX,1090.737502,2026-09-28
5,6,DEN,1074.899213,2026-09-28
6,7,NE,1055.545798,2026-09-28
7,8,PHI,1052.158872,2026-09-28
8,9,LA,1048.959580,2026-09-28
9,10,KC,1043.150809,2026-09-28


In [45]:
### Save Data
current_ratings.to_csv("current_elo_rankings.csv", index=False)
daily_projections.to_csv("daily_projections.csv", index=False)
weekly_projections.to_csv("weekly_projections.csv", index=False)
future_projections.to_csv("future_projections.csv", index=False)
# historical_elo.to_csv("nfl_elo_history.csv", index=False)